# Coleta dos 10 Ultimos jogos

In [1]:
import pandas as pd
import numpy as np
import pyautogui
import pyperclip
import openpyxl
import re

In [2]:
# Lidar com Tempo de espera
import time
from time import sleep

############################### SELENIUM ###############################
from selenium import webdriver # navegador

# Ações ###############################
from selenium.webdriver.common.by import By # localizar elementos
from selenium.webdriver.common.keys import Keys # comandos do teclado
from selenium.webdriver.common.action_chains import ActionChains # ações do mouse e teclado

# Exceções|Erros e Espera ###############################
from selenium.common.exceptions import NoSuchElementException # exceção para elementos não encontrados # CONTROLE DE ERROS
from selenium.webdriver.support.ui import WebDriverWait # esperar

# condições de espera ###############################
from selenium.webdriver.support.expected_conditions import (visibility_of, staleness_of, invisibility_of_element, visibility_of_element_located)
from selenium.webdriver.support import expected_conditions as EC # condições de espera (atalho)
from selenium.common.exceptions import TimeoutException # exceção para tempo limite

## ChromeDriver ###############################
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
browser = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

# Abrindo site
browser.get('https://www.365scores.com/pt-br')

In [3]:
caminho = 'C:/Users/Raphael/OneDrive/Documentos/GitHub/World_Cup_2026/Coleta de Dados/Dados em CSV/'
df = pd.read_excel('C:/Users/raphael.eugenio/Desktop/Raphael/WC26/paises.xlsx')

print(df.head())

                   Time Confederação
0  Bósnia e Herzegovina         UEFA
1                 Suíça         UEFA
2               Escócia         UEFA
3               Turquia         UEFA
4              Alemanha         UEFA


# 1 SELEÇÃO

In [4]:
selecao = "Brasil"

In [ ]:
pyautogui.position()

In [ ]:
# Botão de Pesquisa
browser.find_element("xpath", "/html/body/div[2]/div/div/div[1]/header/div[1]/div[2]/button").click()
time.sleep(1)

# Botão de inserir nomes(input)
browser.find_element("xpath", "/html/body/div[2]/div/div/div[3]/div/div[1]/div/div[2]/input").send_keys(selecao)

# Escolhendo a seleção
try:
    browser.find_element("xpath", "/html/body/div[2]/div/div/div[3]/div/div[2]/div[2]/div[2]/div[2]/div[1]/div[1]/a/div/div[2]").click()
    time.sleep(1)
except:
    pyautogui.click(x=1192, y=311)
    time.sleep(2)
    
# Escolhendo os resultados
browser.find_element("xpath", '//*[@id="sideBarModule_newCompetitor"]/div[1]/div[1]/div[2]').click()
time.sleep(1)

##### Primeira Partida ##########
pyautogui.click(x=426, y=611) 

# Selecionando as estatísticas
browser.find_element("xpath", '//*[@id="navigation-tabs_game-center_stats"]/div').click()
time.sleep(3)

# Copiando todos os dados
pyautogui.hotkey("ctrl","a")
time.sleep(0.5)
pyautogui.hotkey("ctrl", "c")

time.sleep(0.5)
texto = pyperclip.paste()

# ----------------------------
# 1. PEGAR SÓ A PARTE DO JOGO ATUAL
# ----------------------------
if "Top Stats" in texto:
    parte = texto.split("Top Stats")[0]
else:
    parte = texto

linhas = [l.strip() for l in parte.split('\n') if l.strip() != ""]

# ----------------------------
# 2. ACHAR TIMES DO JOGO (National Team não precisa ser consecutivo)
# ----------------------------
times_encontrados = []
for linha in linhas:
    if "National Team" in linha:
        nome_limpo = linha.replace(" National Team", "").strip()
        if nome_limpo not in times_encontrados:
            times_encontrados.append(nome_limpo)
    if len(times_encontrados) == 2:
        break

time1 = times_encontrados[0] if len(times_encontrados) > 0 else None
time2 = times_encontrados[1] if len(times_encontrados) > 1 else None

print("Times:", time1, "x", time2)

# ----------------------------
# 3. DESCOBRIR LADO DA SELEÇÃO PESQUISADA
# ----------------------------
# time1 = aparece PRIMEIRO no texto = lado ESQUERDO nas stats
if time1 == selecao:
    lado_selecao = "Esquerda"
elif time2 == selecao:
    lado_selecao = "Direita"
else:
    lado_selecao = None
    print(f"AVISO: '{selecao}' não encontrado. Times detectados: {time1} x {time2}")

print(f"{selecao} está na:", lado_selecao)

# ----------------------------
# 4. PEGAR SÓ AS ESTATÍSTICAS
# ----------------------------
stats_texto = texto.split("Top Stats")[1]

linhas = [l.strip() for l in stats_texto.split('\n') if l.strip() != ""]

dados = []

i = 0
while i < len(linhas) - 2:
    
    if re.match(r'^[\d./()%]+$', linhas[i]) and not re.match(r'^[\d./()%]+$', linhas[i+1]):
        
        valor_esq = linhas[i]
        nome = linhas[i+1]
        valor_dir = linhas[i+2] if re.match(r'^[\d./()%]+$', linhas[i+2]) else None
        
        dados.append({
            'Metrica': nome,
            'Esquerda': valor_esq,
            'Direita': valor_dir
        })
        
        i += 3
    else:
        i += 1

df = pd.DataFrame(dados)

# ----------------------------
# 5. PEGAR SÓ A SELEÇÃO PESQUISADA
# ----------------------------
if lado_selecao == "Esquerda":
    df_selecao = df[['Metrica', 'Esquerda']].rename(columns={'Esquerda': 'Valor'})
elif lado_selecao == "Direita":
    df_selecao = df[['Metrica', 'Direita']].rename(columns={'Direita': 'Valor'})
else:
    df_selecao = pd.DataFrame()  # fallback vazio

df_selecao.insert(0, 'Selecao', selecao)
print(df_selecao)
df_selecao.to_excel(selecao + ".xlsx", index=False)

Times: Brasil x Croácia
Brasil está na: Esquerda
   Selecao                               Metrica Valor
0   Brasil                         Posse de Bola   45%
1   Brasil                   Gols esperados (xG)  2.19
2   Brasil                       Total de chutes    13
3   Brasil                         Chutes no gol     7
4   Brasil             Chances perigosas criadas     4
5   Brasil                            Escanteios     2
6   Brasil                          Impedimentos     1
7   Brasil                      Passes completos   403
8   Brasil                     Cartões vermelhos     0
9   Brasil                                Ataque   133
10  Brasil                   Gols esperados (xG)  2.19
11  Brasil                       Total de chutes    13
12  Brasil                         Chutes no gol     7
13  Brasil                      Chutes para fora     1
14  Brasil                     Chutes bloqueados     5
15  Brasil                                 Trave     0
16  Brasil      

In [13]:
df_brasil

,Metrica,Valor
0,Posse de Bola,55%
1,Gols esperados (xG),0.96
2,Total de chutes,12
3,Chutes no gol,3
4,Chances perigosas criadas,1
5,Escanteios,2
6,Impedimentos,0
7,Passes completos,498
8,Cartões vermelhos,0
9,Ataque,155
